In [4]:
# =============================================================================
# STEP 1 - IMMUTABLE RESULTS LEDGER
#
# Figures 2 and 3 went stale, and Figure 3 rendered a literal "nan", because every
# headline number was typed independently into prose, captions and plotting code.
# There was no single source any of them derived from. This notebook builds that
# source: one JSON regenerated from the committed result files, plus a checker that
# scans the manuscript and flags any number that disagrees with it.
#
# Nothing here computes new science. It reads what is already committed.
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess, hashlib, re
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, conformal
import numpy as np, pandas as pd
from scipy import stats
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY
RNG=np.random.default_rng(20260726); B=4000
print('ready:', os.getcwd(), '| alpha', ALPHA)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research | alpha 0.05


In [5]:
# =============================================================================
# Cell 2 - helpers. Every ledger entry carries its provenance: which file it came
# from, so a disputed number can be traced without re-deriving it.
# =============================================================================
LEDGER={'_meta':{'alpha_primary':ALPHA,'generated_from':'committed reports/','entries':{}}}

def put(key, value, source, note=None):
    """Record a value with its source file."""
    d=LEDGER
    parts=key.split('.')
    for p in parts[:-1]: d=d.setdefault(p,{})
    d[parts[-1]]=value
    LEDGER['_meta']['entries'][key]={'source':source, **({'note':note} if note else {})}

def seed_ci(df, col='coverage', clus='seed'):
    cl=df[clus].unique()
    if len(cl)<2: return (float(df[col].mean()), None, None)
    means=np.array([df[df[clus]==c][col].mean() for c in cl])
    bs=np.array([RNG.choice(means,len(means),replace=True).mean() for _ in range(B)])
    return (float(means.mean()), float(np.percentile(bs,2.5)), float(np.percentile(bs,97.5)))

def r4(x): return None if x is None else round(float(x),4)
print('helpers ready')


helpers ready


In [6]:
# =============================================================================
# Cell 3 - coverage, per environment.
#
# The four coverage files do NOT share a schema, so every access goes through a
# resolver rather than assuming a column name. Confirmed differences:
#   set size   : NSL uses 'mean_set_size', the other three use 'set_size'
#   rung       : present only on the two laddered datasets (NSL, CIC-IoT)
#   realization: UGR uses 'environment' instead
#   alpha      : the NSL primary file holds only alpha=0.05, so alpha-sensitivity
#                for NSL/CIC/UGR is read from alpha_sensitivity.csv instead
# NSL's file is already filtered to APS/Mondrian and feasible rows (verified), so
# no further filtering is applied; feasible rows are honoured where the flag exists.
# =============================================================================
def col(d, *names):
    for n in names:
        if n in d.columns: return n
    return None

ENV={'nslkdd':      ('coverage_primary_nslkdd.csv','R2L'),
     'cicids2017':  ('coverage_primary_cicids2017.csv','DoS'),
     'ugr16':       ('coverage_primary_ugr16.csv','nerisbotnet'),
     'ciciot2023':  ('coverage_primary_ciciot2023.csv','Web')}

for ds,(f,focal) in ENV.items():
    d=pd.read_csv(RD/f)
    if 'feasible' in d.columns: d=d[d['feasible']]
    ss=col(d,'set_size','mean_set_size'); rk=col(d,'rung')
    put(f'coverage.{ds}.focal_class', focal, f)
    put(f'coverage.{ds}.schema', {'set_size_col':ss,'has_rung':bool(rk),
        'realization_col':col(d,'realization','environment'),
        'alphas_in_file':sorted(float(a) for a in d.alpha.unique())}, f)
    p_=d[np.isclose(d.alpha,ALPHA)]
    for proto in ['REC','TSC','SHC']:
        g=p_[(p_['class']==focal)&(p_.protocol==proto)]
        if not len(g): continue
        m,lo,hi=seed_ci(g)
        put(f'coverage.{ds}.focal.{proto}.pooled', {'mean':r4(m),'ci':[r4(lo),r4(hi)]}, f)
        if rk:
            by={}
            for r in sorted(g[rk].unique()):
                gr=g[np.isclose(g[rk],r)]; mm,l2,h2=seed_ci(gr)
                by[f'{float(r):.2f}']={'mean':r4(mm),'ci':[r4(l2),r4(h2)]}
            put(f'coverage.{ds}.focal.{proto}.by_rung', by, f)
    put(f'coverage.{ds}.all_classes_SHC',
        {c:r4(g['coverage'].mean()) for c,g in p_[p_.protocol=='SHC'].groupby('class')}, f)
    if ss:
        put(f'coverage.{ds}.set_size',
            {pr:r4(gg[ss].mean()) for pr,gg in p_.groupby('protocol')}, f, f'column {ss}, pooled')
        # the manuscript cites rung-specific set sizes, so record those too
        if rk:
            fs={}
            for r in sorted(p_[rk].unique()):
                gr=p_[(np.isclose(p_[rk],r))&(p_['class']==focal)]
                fs[f'{float(r):.2f}']={pr:r4(gg[ss].mean()) for pr,gg in gr.groupby('protocol')}
            put(f'coverage.{ds}.focal_set_size_by_rung', fs, f, f'column {ss}, focal class only')
        fsz={pr:r4(gg[ss].mean()) for pr,gg in p_[p_['class']==focal].groupby('protocol')}
        put(f'coverage.{ds}.focal_set_size', fsz, f, f'column {ss}, focal class only')
    print(f'{ds:12s} focal {focal:12s} SHC {LEDGER["coverage"][ds]["focal"]["SHC"]["pooled"]["mean"]}'
          f' | set-size col {ss} | rung {bool(rk)}')

# ---- alpha sensitivity: NSL/CIC/UGR from alpha_sensitivity.csv, CIC-IoT from its coverage file
if (RD/'alpha_sensitivity.csv').exists():
    a=pd.read_csv(RD/'alpha_sensitivity.csv')
    key={'nslkdd':'nslkdd','cicids2017':'cicids2017','ugr16':'ugr16'}
    for ds,tok in key.items():
        g=a[a.dataset.str.startswith(tok)]
        if len(g):
            put(f'coverage.{ds}.focal_SHC_by_alpha',
                {f'{float(r.alpha):.2f}':r4(r.SHC) for _,r in g.iterrows()}, 'alpha_sensitivity.csv')
d=pd.read_csv(RD/'coverage_primary_ciciot2023.csv')
if 'feasible' in d.columns: d=d[d['feasible']]
rk=col(d,'rung')
top=d[rk].max() if rk else None
al={}
for aa in sorted(d.alpha.unique()):
    g=d[(np.isclose(d.alpha,aa))&(d['class']=='Web')&(d.protocol=='SHC')]
    if rk is not None: g=g[np.isclose(g[rk],top)]
    if len(g): al[f'{float(aa):.2f}']=r4(g['coverage'].mean())
put('coverage.ciciot2023.focal_SHC_by_alpha', al, 'coverage_primary_ciciot2023.csv',
    f'top rung {top}')
print('alpha-sensitivity recorded for all four')

# ---- UGR discovered classes
u=pd.read_csv(RD/'coverage_primary_ugr16.csv'); u=u[np.isclose(u.alpha,ALPHA)&(u.protocol=='SHC')]
for c in ['scan11','scan44']:
    g=u[u['class']==c]
    if len(g):
        m,lo,hi=seed_ci(g)
        put(f'coverage.ugr16.discovered.{c}', {'mean':r4(m),'ci':[r4(lo),r4(hi)]},
            'coverage_primary_ugr16.csv')
print('discovered UGR classes recorded')


nslkdd       focal R2L          SHC 0.086 | set-size col mean_set_size | rung True
cicids2017   focal DoS          SHC 0.6039 | set-size col set_size | rung False
ugr16        focal nerisbotnet  SHC 0.9473 | set-size col set_size | rung False
ciciot2023   focal Web          SHC 0.9511 | set-size col set_size | rung True
alpha-sensitivity recorded for all four
discovered UGR classes recorded


In [7]:
# =============================================================================
# Cell 4 - shift measures, mechanism, monitor, controls, efficiency, models.
# =============================================================================
# shift
for ds,f in [('nslkdd','ladder_shift_measures_nslkdd.csv'),
             ('cicids2017','ladder_shift_measures_cicids2017.csv'),
             ('ciciot2023','ladder_shift_measures_ciciot2023.csv')]:
    if (RD/f).exists():
        s=pd.read_csv(RD/f)
        put(f'shift.{ds}', {k:[r4(s[k].min()),r4(s[k].max())] for k in ['S_cov','S_lab','S_sup'] if k in s}, f)
if (RD/'ladder_shift_measures_ugr16.json').exists():
    j=json.loads((RD/'ladder_shift_measures_ugr16.json').read_text())
    put('shift.ugr16', {k:j[k] for k in ('S_cov','S_lab','S_sup') if k in j}, 'ladder_shift_measures_ugr16.json')

# mechanism, class level, one rung per laddered dataset
m=pd.read_csv(RD/'score_shift_explainability.csv'); m['ds']=m['dataset'].str.split(':').str[0]
cl=m.groupby(['dataset','class','ds'],as_index=False)[['score_KS','undercoverage']].mean()
rho,p_=stats.spearmanr(cl.score_KS, cl.undercoverage)
bs=[]
for _ in range(B):
    s=cl.iloc[RNG.choice(len(cl),len(cl),replace=True)]
    if s.score_KS.nunique()>2: bs.append(stats.spearmanr(s.score_KS,s.undercoverage)[0])
put('mechanism.class_level', {'rho':r4(rho),'p':float(p_),'n':int(len(cl)),
    'ci':[r4(np.percentile(bs,2.5)),r4(np.percentile(bs,97.5))]}, 'score_shift_explainability.csv')
put('mechanism.cells_per_dataset', cl.groupby('ds').size().to_dict(), 'score_shift_explainability.csv')
put('mechanism.range', {'score_KS':[r4(cl.score_KS.min()),r4(cl.score_KS.max())],
                        'undercoverage':[r4(cl.undercoverage.min()),r4(cl.undercoverage.max())]},
    'score_shift_explainability.csv')
per={}; loo={}
for ds,g in cl.groupby('ds'):
    if len(g)>3:
        r,pp=stats.spearmanr(g.score_KS,g.undercoverage); per[ds]={'rho':r4(r),'p':r4(pp),'n':int(len(g))}
    s=cl[cl.ds!=ds]; r2,_=stats.spearmanr(s.score_KS,s.undercoverage); loo[ds]=r4(r2)
put('mechanism.within_dataset', per, 'score_shift_explainability.csv')
put('mechanism.leave_one_out', loo, 'score_shift_explainability.csv',
    'rho with each dataset removed in turn')

# monitor (feasible-only, within-environment ranks)
mo=pd.read_csv(RD/'monitor_labelfree.csv'); mo['ds']=mo['dataset'].str.split(':').str[0]
fb=pd.read_csv(RD/'feasibility_binding_nslkdd.csv')
infeas=set(fb[(np.isclose(fb.alpha,ALPHA))&(~fb.shc_feasible)]['class'])
mo=mo[~((mo.ds=='nslkdd')&(mo['class'].isin(infeas)))].reset_index(drop=True)
det=np.full(len(mo),np.nan)
for ds,g in mo.groupby('ds'):
    rd=g['drift_labelfree'].rank(pct=True); rm=g['pred_mass_drop'].clip(lower=0).rank(pct=True)
    det[g.index]=np.where(g['low_support'].fillna(False),rm,rd)
mo['det']=det; ok=~np.isnan(mo.det)
from sklearn.metrics import roc_auc_score
y=(mo.loc[ok,'undercoverage']>0.05).astype(int)
rr,_=stats.spearmanr(mo.loc[ok,'det'], mo.loc[ok,'undercoverage'])
put('monitor.pooled', {'rho':r4(rr),'auroc':r4(roc_auc_score(y,mo.loc[ok,'det'])),
    'n':int(ok.sum()),'n_undercovering':int(y.sum())}, 'monitor_labelfree.csv',
    'feasible cells only; ranks computed within environment (deployable)')
pm={}
for ds,g in mo[ok].groupby('ds'):
    yy=(g.undercoverage>0.05).astype(int)
    if yy.nunique()>1:
        r,_=stats.spearmanr(g.det,g.undercoverage)
        pm[ds]={'rho':r4(r),'auroc':r4(roc_auc_score(yy,g.det)),'n':int(len(g)),'n_under':int(yy.sum())}
put('monitor.per_dataset', pm, 'monitor_labelfree.csv')

# negative control
nc=pd.read_csv(RD/'negative_control.csv')
put('negative_control.nslkdd', nc.to_dict('records'), 'negative_control.csv')
put('negative_control.max_abs_deviation', r4((nc['coverage']-(1-ALPHA)).abs().max()), 'negative_control.csv')

# efficiency, model performance, calibration
if (RD/'efficiency_setsize.csv').exists():
    e=pd.read_csv(RD/'efficiency_setsize.csv')
    put('efficiency', e.to_dict('records'), 'efficiency_setsize.csv')
mp={}
for ds,f in [('nslkdd','model_performance.csv'),('cicids2017','model_performance_cicids2017.csv'),
             ('ugr16','model_performance_ugr16.csv'),('ciciot2023','model_performance_ciciot2023.csv')]:
    if (RD/f).exists():
        d=pd.read_csv(RD/f); col=[c for c in d.columns if 'f1' in c.lower()][-1]
        mp[ds]={a:r4(g[col].mean()) for a,g in d.groupby('arch')}
put('model_performance.macro_f1', mp, 'model_performance*.csv')
if (RD/'calibration_quality.csv').exists():
    cq=pd.read_csv(RD/'calibration_quality.csv')
    put('calibration.mean_ece_by_dataset', {k:r4(v) for k,v in cq.groupby('dataset')['ece'].mean().items()},
        'calibration_quality.csv')
    put('calibration.max_per_class_ece', r4(cq.ece.max()), 'calibration_quality.csv')
print('ledger entries:', len(LEDGER['_meta']['entries']))


ledger entries: 63


In [8]:
# =============================================================================
# Cell 5 - the remaining result families. The manuscript cites these and the
# ledger must contain them, otherwise the checker reports them as unverified and
# the whole exercise is decorative.
# =============================================================================
def maybe(f):
    q=RD/f
    return pd.read_csv(q) if q.exists() and q.suffix=='.csv' else (
           json.loads(q.read_text()) if q.exists() else None)

# guarantee-band shortfalls
g=maybe('coverage_guarantee_band.csv')
if g is not None:
    sh={}
    for _,r in g[(g.protocol=='SHC')&(g.below_band)].iterrows():
        sh.setdefault(r['dataset'],{})[r['class']]=r4(r['violation_size'])
    put('guarantee_band.shc_shortfall', sh, 'coverage_guarantee_band.csv')
    put('guarantee_band.above_band', g[g.above_band][['dataset','class','protocol','observed']]
        .to_dict('records'), 'coverage_guarantee_band.csv')

# marginal vs class-conditional
mm=maybe('marginal_vs_mondrian_nslkdd.csv')
if mm is not None: put('marginal_vs_mondrian.nslkdd', mm.to_dict('records'),
                       'marginal_vs_mondrian_nslkdd.csv')

# inverse-probability weighting
ab=maybe('armb_verdict.json')
if ab: put('ipw', {k:v for k,v in ab.items() if k in ('cicids2017','ugr16')}, 'armb_verdict.json')

# selective prediction
sp=maybe('selective_prediction_curve.csv')
if sp is not None:
    put('selective.by_dataset', {ds:{f'{float(r.escalated_frac):.3f}':
        {'focal':r4(r.focal_cov_retained),'marginal':r4(r.marginal_cov_retained),
         'retained':r4(r.focal_retained_frac)} for _,r in gg.iterrows()}
        for ds,gg in sp.groupby('dataset')}, 'selective_prediction_curve.csv')

# per-rung shift measures (NSL table cites these)
ns=maybe('ladder_shift_measures_nslkdd.csv')
if ns is not None:
    put('shift.nslkdd_by_rung', {f'{float(r):.2f}':{k:r4(v) for k,v in gg[['S_cov','S_lab','S_sup']].mean().items()}
        for r,gg in ns.groupby('rung')}, 'ladder_shift_measures_nslkdd.csv')
ci=maybe('ladder_shift_measures_ciciot2023.csv')
if ci is not None:
    put('shift.ciciot2023_by_rung', {f'{float(r):.2f}':{k:r4(v) for k,v in gg[['S_cov','S_lab','S_sup']].mean().items()}
        for r,gg in ci.groupby('rung')}, 'ladder_shift_measures_ciciot2023.csv')

# monitor per dataset (as published) and the boundary
mpd=maybe('monitor_per_dataset.csv')
if mpd is not None: put('monitor.published_per_dataset', mpd.to_dict('records'), 'monitor_per_dataset.csv')
mv=maybe('monitor_verdict.json')
if mv: put('monitor.boundary', {k:v for k,v in mv.items() if 'boundary' in k.lower() or 'misroute' in k.lower()},
           'monitor_verdict.json')

# score movement detail, all datasets
ss=maybe('score_shift_explainability.csv')
if ss is not None:
    ss=ss.copy(); ss['ds']=ss['dataset'].str.split(':').str[0]
    put('score_movement.by_dataset_class',
        {f"{r['ds']}|{r['class']}":{'score_KS':r4(r['score_KS']),'undercoverage':r4(r['undercoverage'])}
         for _,r in ss.groupby(['ds','class'],as_index=False)[['score_KS','undercoverage']].mean().iterrows()},
        'score_shift_explainability.csv')
iv=maybe('ciciot2023_mechanism_verdict.json')
if iv: put('score_movement.ciciot2023', {k:v for k,v in iv.items() if isinstance(v,(int,float,str))},
           'ciciot2023_mechanism_verdict.json')
fm=maybe('focal_score_movement_ciciot2023.csv')
if fm is not None:
    put('score_movement.ciciot2023_by_rung',
        {f'{float(r):.2f}':{k:r4(v) for k,v in gg[['KS_all','KS_novel','KS_seen','misroute_all']].mean().items()}
         for r,gg in fm.groupby('rung')}, 'focal_score_movement_ciciot2023.csv')

# architecture, pooled model, specification curve, feature audit
pa=maybe('per_architecture_coverage.csv')
if pa is not None: put('architecture.focal_coverage', pa.to_dict('records'), 'per_architecture_coverage.csv')
pm=maybe('pooled_model_results.json')
if pm: put('pooled_model', {k:(v if not isinstance(v,dict) else {kk:vv for kk,vv in v.items()})
           for k,v in pm.items() if k.startswith('beta') or k in ('n_rows','n_units')}, 'pooled_model_results.json')
sc=maybe('specification_curve.csv')
if sc is not None:
    put('specification_curve', {'n_specs':int(len(sc)),'alphas':sorted(float(a) for a in sc.alpha.unique()),
        'gap_range':[r4(sc.gap.min()),r4(sc.gap.max())],
        'n_undercovering':int((sc.coverage < sc.nominal-0.02).sum())}, 'specification_curve.csv')
au=maybe('audit_corrections.json')
if au: put('audit', {k:v for k,v in au.items() if k.startswith(('C1','C5','C6'))}, 'audit_corrections.json')

# CIC-IoT provenance figures the manuscript quotes
fr=maybe('focal_class_record_ciciot2023.json')
if fr: put('ciciot2023.provenance', {k:fr[k] for k in
      ('release_files','release_rows','release_labels','working_frame_rows','focal_class') if k in fr},
      'focal_class_record_ciciot2023.json')
sr=maybe('ciciot2023_split_record.json')
if sr: put('ciciot2023.split', {k:sr[k] for k in
      ('novel_subtypes','focal_source_cal_pool','focal_quota_per_eval','eval_n') if k in sr},
      'ciciot2023_split_record.json')
if fr and 'prior_after_cap' in fr:
    put('ciciot2023.prior_after_cap', {k:r4(v) for k,v in fr['prior_after_cap'].items()},
        'focal_class_record_ciciot2023.json')

# quantities the manuscript cites that no earlier block captured
if 'monitor' in LEDGER and 'pooled' in LEDGER['monitor']:
    mo2=maybe('monitor_labelfree.csv')
    if mo2 is not None:
        from sklearn.metrics import roc_auc_score as _auc
        mo2['ds']=mo2['dataset'].str.split(':').str[0]
        fb2=maybe('feasibility_binding_nslkdd.csv')
        inf2=set(fb2[(np.isclose(fb2.alpha,ALPHA))&(~fb2.shc_feasible)]['class']) if fb2 is not None else set()
        mo2=mo2[~((mo2.ds=='nslkdd')&(mo2['class'].isin(inf2)))].reset_index(drop=True)
        dd=np.full(len(mo2),np.nan)
        for ds,g in mo2.groupby('ds'):
            rd=g['drift_labelfree'].rank(pct=True); rm=g['pred_mass_drop'].clip(lower=0).rank(pct=True)
            dd[g.index]=np.where(g['low_support'].fillna(False),rm,rd)
        mo2['det']=dd; okm=~np.isnan(mo2.det)
        yy=(mo2.loc[okm,'undercoverage']>0.05).astype(int); dv=mo2.loc[okm,'det'].to_numpy()
        bs=[]
        for _ in range(B):
            ii=RNG.choice(len(dv),len(dv),replace=True)
            if len(np.unique(yy.to_numpy()[ii]))>1: bs.append(_auc(yy.to_numpy()[ii],dv[ii]))
        put('monitor.pooled_auroc_ci', [r4(np.percentile(bs,2.5)),r4(np.percentile(bs,97.5))],
            'monitor_labelfree.csv')

sd=maybe('selective_prediction_headline.csv')
if sd is not None: put('selective.headline', sd.to_dict('records'), 'selective_prediction_headline.csv')

# permutation-null threshold for S_cov, cited in Section 3
for f in ['permutation_null.json','ladder_shift_measures_ugr16.json','scov_permutation_null.json']:
    j=maybe(f)
    if isinstance(j,dict):
        for k,v in j.items():
            if 'null' in k.lower() and isinstance(v,(int,float)):
                put('shift.permutation_null_threshold', r4(v), f); break

# NSL dose-response slope, cited in 5.3
au2=maybe('audit_corrections.json')
if isinstance(au2,dict):
    for k,v in au2.items():
        if isinstance(v,dict) and 'slope' in json.dumps(v).lower():
            put('dose_response.nslkdd', v, 'audit_corrections.json'); break

# CIC-IoT prior as a percentage, since the manuscript quotes 1.64 per cent
fr2=maybe('focal_class_record_ciciot2023.json')
if isinstance(fr2,dict) and 'prior_after_cap' in fr2:
    put('ciciot2023.focal_prior_percent', r4(100*fr2['prior_after_cap'].get('Web',0)),
        'focal_class_record_ciciot2023.json')

print('ledger entries after expansion:', len(LEDGER['_meta']['entries']))


ledger entries after expansion: 86


In [9]:
# =============================================================================
# Cell 6 - write the ledger, then CHECK THE MANUSCRIPT AGAINST IT.
# The checker extracts every number of the form 0.xxxx from the manuscript and
# reports which appear in the ledger and which do not. A number absent from the
# ledger is not necessarily wrong, but it is unverified, and that is the state
# that produced the stale figures.
# =============================================================================
LEDGER['_meta']['n_entries']=len(LEDGER['_meta']['entries'])
out=RD/'final_results.json'
out.write_text(json.dumps(LEDGER, indent=2, default=str))
print('wrote', out, f'({out.stat().st_size/1000:.1f} kB, {len(LEDGER["_meta"]["entries"])} keyed entries)')

def flatten(d, pre=''):
    vals=set()
    if isinstance(d,dict):
        for k,v in d.items():
            if k=='_meta': continue
            vals |= flatten(v, f'{pre}.{k}')
    elif isinstance(d,(list,tuple)):
        for v in d: vals |= flatten(v, pre)
    elif isinstance(d,(int,float)) and d is not None:
        vals.add(round(float(d),4))
    return vals
LV=flatten(LEDGER)
print(f'distinct numeric values in ledger: {len(LV)}')

MS=PROJECT_ROOT/'FULL_MANUSCRIPT.md'
if not MS.exists():
    print(f'\nmanuscript not found at {MS}; place FULL_MANUSCRIPT.md in the repo root to enable checking')
else:
    md=MS.read_text()
    body=md[:md.index('# References')] if '# References' in md else md
    # drop markdown headings: '## 4.11 Evaluation...' would otherwise register 4.11
    body='\n'.join(l for l in body.split('\n') if not l.lstrip().startswith('#'))
    # Catch decimals of any magnitude, not just 0.xxx: set sizes (2.04, 4.68),
    # slopes (2.071) and correlations above 1 were all invisible to the earlier
    # 0.xxx-only pattern. Exclude structural references and years.
    STRUCT=re.compile(r'(Section|Table T|Figure|Amendment|Appendix)\s*$')
    nums=set()
    for m in re.finditer(r'(?<![\w.])(\d{1,3}\.\d{2,4})(?![\d])', body):
        pre=body[max(0,m.start()-14):m.start()]
        if STRUCT.search(pre): continue                 # Section 5.13, Table T1.2 etc
        v=float(m.group(1))
        if 1900 <= v <= 2100 and '.' not in m.group(1)[:4]: continue
        nums.add(round(v,4))
    # Three tiers. Only UNMATCHED is the failure mode that produced the stale figures;
    # ROUNDED is a citation at lower precision and is acceptable.
    exact=[n for n in nums if any(abs(n-v)<=0.0011 for v in LV)]
    rounded=[n for n in nums if n not in exact and any(abs(n-v)<=0.006 for v in LV)]
    unverified=sorted(n for n in nums if n not in exact and n not in rounded)
    print(f'\ndecimal values in the manuscript body: {len(nums)}')
    print(f'  exact match to ledger   : {len(exact)}')
    print(f'  rounded match (<=0.006) : {len(rounded)}  {sorted(rounded)}')
    print(f'  UNMATCHED               : {len(unverified)}')
    for n in unverified:
        ctx=[body[max(0,m.start()-70):m.start()+25].replace("\n"," ")
             for m in re.finditer(re.escape(f'{n:.4f}'.rstrip("0").rstrip(".")), body)][:1]
        print(f'   {n}  {ctx[0] if ctx else ""}')
    if unverified:
        print('\n  An unmatched number is not necessarily wrong, but it is not derived from any')
        print('  committed result. Every stale figure in this project began in that state.')
    (RD/'manuscript_number_check.txt').write_text(
        f'ledger entries {len(LEDGER["_meta"]["entries"])}, distinct values {len(LV)}\n'
        f'manuscript numbers {len(nums)}: exact {len(exact)}, rounded {len(rounded)}, '
        f'unmatched {len(unverified)}\n' + '\n'.join(str(n) for n in unverified))
    print('\nwrote reports/manuscript_number_check.txt')


wrote /content/drive/MyDrive/CALSHIFT_Research/calshift-research/reports/final_results.json (50.7 kB, 86 keyed entries)
distinct numeric values in ledger: 602

manuscript not found at /content/drive/MyDrive/CALSHIFT_Research/calshift-research/FULL_MANUSCRIPT.md; place FULL_MANUSCRIPT.md in the repo root to enable checking


In [ ]:
# =============================================================================
# Cell 7 - commit
# =============================================================================
def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','step 1: immutable results ledger (final_results.json) regenerated from committed reports, plus manuscript number checker')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)
